In [ ]:
from pyspark.sql.session import SparkSession
from pyspark.sql.functions import *

spark = (
    SparkSession
    .builder
    .master("local[*]")
    .appName("intro_delta")
    .config(
        "spark.jars.packages",
        "org.apache.hadoop:hadoop-aws:3.3.4,"
        "com.amazonaws:aws-java-sdk-bundle:1.12.261,"
        "io.delta:delta-spark_2.12:3.3.0,"
        "org.scala-lang:scala-library:2.12.18"
    )
    # Configurações essenciais do Delta
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    # Configurações do S3/MinIO
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.endpoint", "http://localhost:9000")
    .config("spark.hadoop.fs.s3a.access.key", "minioadmin")
    .config("spark.hadoop.fs.s3a.secret.key", "minioadmin")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    # .config("spark.driver.memory", "12g") 
    # .config("spark.executor.memory", "12g") 
    # .config("spark.executor.memoryOverhead", "4g")
    # Configurações de compatibilidade
    .config("spark.driver.extraJavaOptions", "-Dio.netty.tryReflectionSetAccessible=true")
    .config("spark.executor.extraJavaOptions", "-Dio.netty.tryReflectionSetAccessible=true")
    .config("spark.databricks.delta.schema.autoMerge.enabled", "true")
    .config("spark.sql.warehouse.dir", "s3a://sparkdw")
    .config("spark.sql.parquet.datetimeRebaseModeInWrite", "CORRECTED")
    .getOrCreate()
)
spark

In [ ]:
dataframe = spark.read.format("parquet").load("/home/doug/project/spark/jobs/notebooks/part-00000-40be3905-1643-4b96-8abc-31502a864176-c000.snappy (1).parquet")

In [ ]:
colors_df = dataframe.select(collect_set("color").alias("colors")).first()["colors"]

# Passo 2: Criar o padrão regex para substituir as cores
regex_pattern = r"\b(" + "|".join(colors_df) + r")\b"

# Passo 3: Aplicar as transformações no DataFrame
dataframe = (
    dataframe
    .withColumn("productname", regexp_replace(split(col("productname"), ",")[0], regex_pattern, ""))
    .withColumn("productname", regexp_replace(col("productname"), r"\s*-\s*", ""))
)

dataframe

In [ ]:
colorList = dataframe.select(collect_set("color").alias("colors")).first()["colors"]
regex_pattern = r"\b(" + "|".join(colorList) + r")\b"

dataframe = (dataframe
                .withColumn("productname", split(col("productname"), ",")[0])
                .withColumn("productname", regexp_replace(col("productname"), regex_pattern, ""))
                .withColumn("productname", regexp_replace(col("productname"), " -", ""))
            )

print(regex_pattern)
dataframe.show()

In [ ]:
#transform productmodel
dataframe = (dataframe
                .withColumn("productnumber", substring_index(col("productnumber"), "-", 2))
)

In [ ]:
#transform class column
dataframe = (dataframe
                    .withColumn("class", 
                                expr(""" 
                                        case
                                            when productcategoryid is null and productsubcategoryid is null then "Discontinued"
                                            when trim(rtrim(class)) = "L" then "Light"
                                            when trim(rtrim(class)) = "M" then "Medium"
                                            when trim(rtrim(class)) = "H" then "Heavy"
                                        else "Small"
                                        end as class
                                    """)))

In [ ]:
dataframe = (dataframe 
                    .withColumn("size",
                                expr(""" 
                                    case	
                                        when ProductSubcategoryID is null and ProductCategoryID is null then 'Discontinued'
                                        when Size = 'L' then 'Large'
                                        when Size = 'M' then 'Medium'
                                        when Size = 'Xl' then 'Extra Large'
                                        when Size = 'S' then 'Small'
                                        when Size <= '42' then 'Small'
                                        when Size > '42' and Size <= '50' then 'Medium'
                                        when Size > '50' and Size <= '60' then 'Large'
                                        when Size > '60' then 'Extra Large'
                                    else 'Small Piece'
                                    end as size
                                """))
                    )

In [ ]:
dataframe = (dataframe
                    .withColumn("productline",
                                expr(""" 
                                    case	
                                        when ProductSubcategoryID is null and ProductCategoryID is null then 'Discontinued'
                                        when trim(ProductLine) = 'R' then 'Road'
                                        when trim(ProductLine) = 'S' then 'Sport'
                                        when trim(ProductLine) = 'M' then 'Mountain'
                                        when trim(ProductLine) = 'T' then 'Touring'
                                    else 'Accessories'
                                    end as productline
                                """)                           
                    ))

In [ ]:
dataframe = (dataframe 
                .withColumn("color", 
                            expr(""" 
                                case
                                    when ProductSubcategoryID is null and ProductCategoryID is null then 'Discontinued'
                                    when Color is null then 'Multicolor'
                                    when color = 'Multi' then 'Multicolor'
                                    when Color like '%/%' then 'Multicolor'
                                else color
                                end as color
                            """))
                )

In [ ]:
dataframe = (dataframe
                .withColumn("style", 
                            expr(""" 
                                case
                                    when trim(rtrim(style)) = "M" then "Masculine"
                                    when trim(rtrim(style)) = "W" then "Feminine"
                                else "Unissex"
                                end as style
                            """))
                )

In [ ]:
dataframe = (dataframe 
                .withColumn("modelname", 
                            expr(""" 
                                case
                                    when ProductSubcategoryID is null and ProductCategoryID is null then 'Discontinued'
                                    when modelname like 'LL%' then REPLACE(modelname, 'LL', 'Light')
                                    when modelname like 'ML%' then REPLACE(modelname, 'ML', 'Medium')
                                    when modelname like 'HL%' then REPLACE(modelname, 'HL', 'Heavy')
                                else modelname
                                end as modelname
                            """)))

In [ ]:
dataframe = (dataframe 
                    .withColumn("weightunitmeasurecode", 
                              expr("""	
                                   case 
                                        when trim(rtrim(weightunitmeasurecode)) is null then 'Not Available'
                                        when trim(rtrim(weightunitmeasurecode)) = 'G' then 'Grams'
                                        when trim(rtrim(weightunitmeasurecode)) = 'LB' then 'Pounds'
                                   else weightunitmeasurecode
                                   end as weightunitmeasurecode
                              """))
                    )

In [ ]:
dataframe = (dataframe
                    .withColumn("manufacturyclass",
                        expr(""" 
                            case 
                                when DaysToManufacture <= '1' then 'Fast'
                                when DaysToManufacture > '1' and DaysToManufacture <= '3' then 'Normal'
                                else 'Slowly'
                            end as manufactureclass
                        """))
                    )

In [ ]:
dataframe = (dataframe
                    .withColumn("sizemeasure", 
                        expr(""" 
                            case
                                when trim(rtrim(sizeunitmeasurecode)) = "CM" then "Centimeters"
                                else "Not Available"
                            end as sizemeasure
                        """))
                    )

In [ ]:
dataframe = (dataframe
                        .fillna({
                                "productsubcategoryid": 0,
                                "subcategoryname":      "Discontinued",
                                "productcategoryid":    0,
                                "productmodelid":       0,
                                "sellenddate":          "2100-12-31",
                                "productsubcategoryid": 0,
                                "subcategoryname":      "Discontinued",
                                "productcategoryid":    0,
                                "categoryname":         "Discontinued",
                                "productmodelid":       0,
                                "sellenddate":          "2100-12-31",
                                "weight":               0.00})
                        )

In [ ]:
dataframe = (dataframe.select(
                        "ProductID",                "productname",
                        "productnumber",            "productsubcategoryid",
                        "subcategoryname",          "productcategoryid",
                        "categoryname",             "color",
                        "class",                    "size",
                        "productline",              "productmodelid",
                        "modelname",                "style",
                        "listprice",                "standardcost",
                        "manufacturyclass",         "sizemeasure",
                        "safetystocklevel",         "reorderpoint",
                        "weightunitmeasurecode",    "weight",
                        "daystomanufacture",        "sellstartdate",
                        "sellenddate",              "inserted_date",
                        "modified_date",            "eltversion"))

dataframe.where("subcategoryname not like 'Discontinued'").show()